# Analyse Expérimentale Huffman Dynamique

## Fonctions pour génération de graphes

In [ ]:
import math
from collections import Counter
import matplotlib.pyplot as plt
from glob import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
def file_to_text(filename="../compression.txt"):
    """lit un fichier et return son contenu en texte"""
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()
    return text

def load_log_file(filename="../compression.txt"):
    """
    lire un fichier log et en extraire les informations
    Format attendu : input_file;output_file;input_size;output_size;rate;time_ms
    """
    columns = ['input_file', 'output_file', 'input_size', 'output_size', 'compression', 'time_ms']
    df = pd.read_csv(filename, sep=';', names=columns, encoding='utf-8')
    return df

def trouve_infos_compression(filename, logfile="../compression.txt"):
    """
    Trouve les informations de compression (dans fichier de log) pour un fichier donné.
    Compare chemins absolus si possible, sinon fallback sur nom de fichier.
    """
    df = load_log_file(logfile)

    filename_abs = os.path.abspath(filename)
    filename_base = os.path.basename(filename)

    # Normalisation
    df['input_abs'] = df['input_file'].apply(lambda x: os.path.abspath(x) if os.path.exists(x) else None)
    df['input_base'] = df['input_file'].apply(os.path.basename)

    # matcher par chemin absolu
    row = df[df['input_abs'] == filename_abs]

    # fallback sur nom de fichier
    if row.empty:
        row = df[df['input_base'] == filename_base]

    if row.empty:
        return None

    return row['input_size'].iloc[0], row['output_size'].iloc[0], row['compression'].mean() * 100, row['time_ms'].iloc[0]

def calculate_entropy(text):
    """Calcule l'entropie de Shannon d'un texte"""
    if not text:
        return 0.0
    
    char_counts = Counter(text)
    total = len(text)
    entropy = 0.0
    
    for count in char_counts.values():
        p = count / total
        entropy -= p * math.log2(p)
    
    return entropy

def analyze_text(filename, logfile="../compression.txt", verbose=False):
    """
    Analyse complète d'un texte avec notre algorithme Huffman dynamique
    """
    text = file_to_text(filename)

    entropy = calculate_entropy(text)
    char_counts = Counter(text)
    nb_chars = len(char_counts)
    total_chars = len(text)
    
    # Informations sur compression avec notre Huffman
    res = trouve_infos_compression(filename, logfile)
    if res is None:
        if verbose:
            print(f"Erreur analyze_text : le fichier {filename} n'a pas été compressé auparavant")
        return None
    
    input_size, output_size, taux_comp, time_ms = res

    # facteur de compression : taille entrée/taille sortie
    facteur = input_size / output_size
    
    if verbose:
        joli_taux_comp = 100*float(taux_comp)
        print(f"\n{'='*50}")
        print(f"Analyse: {filename}")
        print(f"{'='*50}")
        print(f"Taille fichier original: {input_size} octets")
        print(f"Taille fichier compressé: {output_size} octets")
        print(f"Taille texte: {total_chars} caractères")
        print(f"Nbre de caractères différents (taille alphabet): {nb_chars}")
        print(f"Temps de compression (ms): {time_ms}")
        print(f"Taux compression : {joli_taux_comp:.2f}%")
        print(f"Entropie: {entropy:.4f} bits/caractère")
        print(f"Facteur de compression: {facteur:.2f}")

        # Top 5 caractères
        print(f"\nTop 5 caractères:")
        for char, count in char_counts.most_common(5):
            p = count / total_chars
            char_display = repr(char) if char in ['\n', '\t', ' '] else char
            print(f"  {char_display}: {count:6d} ({p*100:5.2f}%)")
    
    return {
        'name': filename,
        'file_size': input_size,
        'text_size': total_chars,
        'entropy': entropy,
        'taux_comp': taux_comp,
        'alpha_size': nb_chars,
        'time_ms': time_ms    
    }

def analyze_all_texts(dirname, logfile="../compression.txt", verbose=False, recursive=False):
    """
    Lance analyze_text sur tous les fichiers .txt d'un répertoire.

    - Si `recursive` est False, on cherche seulement les .txt dans `dirname`
    - Si `recursive` est True, on parcourt récursivement les sous-répertoires

    Retourne une liste des résultats non-None (analyse réussie).
    """
    results = []

    # Construire un pattern adapté selon recursive
    if recursive:
        pattern = os.path.join(dirname, "**", "*.txt")
    else:
        pattern = os.path.join(dirname, "*.txt")

    for filename in glob(pattern, recursive=recursive):
        if verbose:
            print(f"analyze_all_texts : {filename}")
        result = analyze_text(filename, logfile, verbose)
        if result is not None:
            results.append(result)

    print(f"analyze_all_texts : {len(results)} fichiers analysés")

    return results


In [ ]:
def plot_time_nbchar(results) :
    """tracer graphe temps = f(nb caractères i.e taille texte)"""
    text_size = []
    temps = []

    for r in results:
        if r is None:  # ignorer les analyses échouées
            continue
        text_size.append(r['text_size'])
        temps.append(r['time_ms']/1000)  # convertir ms en s

    plt.figure(figsize=(12, 6))
    plt.scatter(text_size, temps, color='red', edgecolors='black', s=100, alpha=0.7)
    plt.xlabel("Taille du texte (nb caractères)", fontsize=12, fontweight='bold')
    plt.ylabel("Temps de compression (s)", fontsize=12, fontweight='bold')
    plt.title("Temps de compression en fonction de la taille du texte", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_time_nbsymb(results) : 
    """trace graphes temps = f(nb symboles dans alphabet)"""
    alpha_size = []
    temps = []

    for r in results:
        if r is None:  # ignorer les analyses échouées
            continue
        alpha_size.append(r['alpha_size'])
        temps.append(r['time_ms']/1000)  # convertir ms en s

    plt.figure(figsize=(12, 6))
    plt.scatter(alpha_size, temps, color='blue', edgecolors='black', s=100, alpha=0.7)

    plt.xlabel("Taille de l'alphabet", fontsize=12, fontweight='bold')
    plt.ylabel("Temps de compression (s)", fontsize=12, fontweight='bold')
    plt.title("Temps de compression en fonction de la taille de l'alphabet", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_time_size(results):
    """
    Trace graphe temps = f(taille fichier)
    """
    taille = []
    temps = []

    for r in results:
        if r is None:  # ignorer les analyses échouées
            continue
        taille.append(r['file_size'])
        temps.append(r['time_ms']/1000)  # convertir ms en s

    plt.figure(figsize=(12, 6))
    plt.scatter(taille, temps, color='pink', edgecolors='black', s=100, alpha=0.7)

    # pour ne pas avoir de notation scientifique de l'axe des abscisses
    ax = plt.gca()
    ax.ticklabel_format(style='plain', axis='x')

    plt.xlabel("Taille du fichier (octets)", fontsize=12, fontweight='bold')
    plt.ylabel("Temps de compression (s)", fontsize=12, fontweight='bold')
    plt.title("Temps de compression en fonction de la taille du fichier", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_compression_entropy(results):
    """
    Trace taux de compression réel en fonction de l'entropie
    et ajoute la limite théorique de Shannon.
    """
    # Filtrer uniquement les résultats valides
    valid_results = [r for r in results if r is not None and r.get('taux_comp') is not None]

    entropies = [r['entropy'] for r in valid_results]
    rates = [r['taux_comp'] for r in valid_results]

    plt.figure(figsize=(12, 6))
    plt.scatter(entropies, rates, color='orange', edgecolors='black', s=100, alpha=0.7, label='Taux de compression')

    # Limite théorique de Shannon
    shannon_limit = [(h / 8) * 100 for h in entropies]
    plt.plot(entropies, shannon_limit, 'b--', linewidth=2, alpha=0.7, label='Limite Shannon')

    plt.xlabel("Entropie (bits/caractère)", fontsize=12, fontweight='bold')
    plt.ylabel("Taux de compression", fontsize=12, fontweight='bold')
    plt.title("Taux de compression en fonction de l'entropie", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


## Résultats expérimentaux sur fichiers textuels existants (question 8) 
Bibliothèques de textes utilisée : projet Gutenberg, cf rapport pour sources

In [ ]:
# Structure de données pour stocker les infos d'analyse (pour graphes)
results_gutenberg = []

#### Texte Français (Notre Dame de Paris, V. Hugo) -- autour de 15 secondes

In [ ]:
!../compresser ../texts/francais.txt ../texts/bintexts/francais.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/francais.txt", verbose=True))

#### Texte Français 2 (Blaise Pascal) -- autour de 4 secondes

In [ ]:
!../compresser ../texts/Blaise_Pascal/Blaise_Pascal.txt ../texts/bintexts/Blaise_Pascal.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/Blaise_Pascal/Blaise_Pascal.txt", verbose=True))

#### Texte Anglais -- autour de 12 secondes

In [ ]:
!../compresser ../texts/anglais.txt ../texts/bintexts/anglais.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/anglais.txt", verbose=True))

#### Texte Grec -- autour de 45 secondes

In [ ]:
!../compresser ../texts/grec.txt ../texts/bintexts/grec.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/grec.txt", verbose=True))

#### Texte Russe -- autour de 8 secondes

In [ ]:
!../compresser ../texts/russe.txt ../texts/bintexts/russe.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/russe.txt", verbose=True))

#### Texte Chinois -- autour de 5 min

In [ ]:
!../compresser ../texts/chinois.txt ../texts/bintexts/chinois.huff

Analyse du fichier et de sa compression

In [ ]:
results_gutenberg.append(analyze_text("../texts/chinois.txt", verbose=True))

#### Graphes pour des textes en langage naturel

In [ ]:
plot_time_size(results_gutenberg)
plot_time_nbchar(results_gutenberg)
plot_time_nbsymb(results_gutenberg)
plot_compression_entropy(results_gutenberg)

## Résultats expérimentaux sur fichiers textuels générés (question 9)
On se sert à présent de fichiers textuels qu'on génère aléatoirement grâce au script generer.sh 
(cf rapport pour explication plus détaillée de celui-ci)

In [ ]:
!./generer.sh --help

In [ ]:
!./generer.sh 10 random 

In [ ]:
!./generer.sh 5 random0 5000

In [ ]:
!./generer.sh 3 random00 10000 'latin:0.7,emojis:0.2'

In [ ]:
!./generer.sh 1 random000 1000 'latin:0,emojis:0.1,japanese:0.5,korean:0.4'

Analyser tous les textes générés puis tracer des graphes

In [ ]:
results_random = []
results_random = analyze_all_texts("../texts/random", recursive=True)

In [ ]:
plot_time_size(results_random)
plot_time_nbchar(results_random)
plot_time_nbsymb(results_random)
plot_compression_entropy(results_random)

## Résultats expérimentaux sur fichiers non textuels (question 10) ###
Enfin, on fait nos dernières expérimentations en utilisant des fichiers qui contiennent
de l’information sous une autre forme : du code. 

On utilise ici des fichiers .py, .sh, .ipynb, .md : on se servira tout simplement de fichiers de code présent dans ce projet.


In [ ]:
results_code = []

#### generate.py -- autour de 1 seconde

Compression

In [ ]:
!../compresser ../src/generate.py ../texts/bintexts/generate.huff 

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("../src/generate.py", verbose=True))

#### Node.py -- autour de 0.5 seconde

Compression

In [ ]:
!../compresser ../src/Node.py ../texts/bintexts/Node.huff

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("../src/Node.py", verbose=True))

#### utils.py -- autour de 0.5 seconde

Compression

In [ ]:
!../compresser ../src/utils.py ../texts/bintexts/utils.huff

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("../src/utils.py", verbose=True))

#### tests/test.ipynb -- autour de 0.7 seconde

Compression

In [ ]:
!../compresser ../tests/tests.ipynb ../texts/bintexts/tests.huff

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("tests.ipynb", verbose=True))

#### README.md -- autour de 0.3 seconde

Compression

In [ ]:
!../compresser ../README.md ../texts/bintexts/README.huff

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("../README.md", verbose=True))

#### generer.sh -- autour de 0.3 seconde

Compression

In [ ]:
!../compresser ./generer.sh ../texts/bintexts/generer.huff

Analyse du fichier et de sa compression

In [ ]:
results_code.append(analyze_text("./generer.sh", verbose=True))

#### Graphes pour fichiers de code

In [ ]:
plot_time_size(results_code)
plot_time_nbchar(results_code)
plot_time_nbsymb(results_code)
plot_compression_entropy(results_code)

## Graphes complémentaires

### Graphes pour textes en langage naturel ET générés aléatoirement

In [ ]:
results = analyze_all_texts("..", verbose=False, recursive=True)
results_len = analyze_all_texts("../texts/analyse/len", verbose=False, recursive=True)
results_symb = analyze_all_texts("../texts/analyse/symboles", verbose=False, recursive=True)

In [ ]:
print("Taille de l'alphabet fixée, variation de la taille du texte")
plot_time_size(results_len)
plot_time_nbchar(results_len)
# plot_compression_entropy(results_len)
print("Taille de texte fixée, variation de la taille de l'alphabet")
plot_time_nbsymb(results_symb)
plot_compression_entropy(results_symb)
print("Tous les fichiers .txt de tous les types")
plot_time_size(results)
plot_time_nbchar(results)
plot_time_nbsymb(results)
plot_compression_entropy(results)


### Graphes pour textes générés aléatoirement 

#### Taille d'alphabet fixe, variation de la longueur du texte

In [ ]:

# Lire le fichier de compression
data = []
with open('../compression.txt', 'r') as f:
    for line in f:
        # Chercher les lignes qui correspondent aux fichiers analyse/len_*.txt
        if 'analyse/len_' in line and '.txt' in line:
            parts = line.strip().split(';')
            filename = parts[0]
            # Extraire la longueur du nom du fichier (avec ou sans -0)
            match = re.search(r'len_(\d+)(?:-0)?\.txt', filename)
            if match:
                length = int(match.group(1))
                # La dernière valeur est le temps de compression
                compression_time = float(parts[-1])
                data.append((length, compression_time))

# Trier les données par longueur
data.sort(key=lambda x: x[0])

# Séparer les données pour le graphique
lengths = [d[0] for d in data]
times = [d[1] for d in data]

print(f"Nombre de fichiers trouvés: {len(data)}")
if data:
    print(f"Plage de longueurs: {min(lengths)} - {max(lengths)}")
    print(f"Plage de temps: {min(times):.2f} - {max(times):.2f} ms")

# Créer le graphique
plt.figure(figsize=(12, 6))
plt.plot(lengths, times, marker='o', linestyle='-', linewidth=2, markersize=4)
plt.xlabel('Longueur du texte (caractères)', fontsize=12)
plt.ylabel('Temps de compression (ms)', fontsize=12)
plt.title('Temps de compression en fonction de la longueur du texte', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Afficher le graphique
plt.show()


#### Taille de texte fixée, variation de la taille de l'alphabet / du nombre de symboles

In [ ]:
# Lire le fichier de compression
data = []
with open('../compression.txt', 'r') as f:
    for line in f:
        # Chercher les lignes qui correspondent aux fichiers analyse/symboles/sym_*.txt
        if 'analyse/symboles/sym_' in line and '.txt' in line:
            parts = line.strip().split(';')
            filename = parts[0]
            # Extraire la longueur du nom du fichier (avec ou sans -0)
            match = re.search(r'sym_(\d+)(?:-0)?\.txt', filename)
            if match:
                length = int(match.group(1))
                # La dernière valeur est le temps de compression
                compression_time = float(parts[-1])
                data.append((length, compression_time))

# Trier les données par longueur
data.sort(key=lambda x: x[0])

# Séparer les données pour le graphique
lengths = [d[0] for d in data]
times = [d[1] for d in data]

print(f"Nombre de fichiers trouvés: {len(data)}")
if data:
    print(f"Plage de longueurs: {min(lengths)} - {max(lengths)}")
    print(f"Plage de temps: {min(times):.2f} - {max(times):.2f} ms")

# Créer le graphique
plt.figure(figsize=(12, 6))
plt.plot(lengths, times, marker='o', linestyle='-', linewidth=2, markersize=4)
plt.xlabel('Nombre de symboles', fontsize=12)
plt.ylabel('Temps de compression (ms)', fontsize=12)
plt.title('Temps de compression en fonction du nombre de symboles', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Afficher le graphique
plt.show()